# Section 5 Master - Data Ingestion to Vector Store

A complete end-to-end flow: bring raw files in, split them, embed them, and **persist** them to disk for retrieval. We use **Ollama** for embeddings, **RecursiveCharacterTextSplitter** for chunking, and store in both **FAISS** and **Chroma**.

---
## The 4-Step Flow

Think of yourself running a **library**. To make a book searchable you do four jobs:

1. **Ingestion - gather the books**
   Walk to the shelves and pick up the raw material (txt, XML, PDF, pages).    Turn each file into a clean "book" - a `Document`.

2. **Transformation - split into chapters**
   A whole book is too big to read at once. Chop it into small, self-contained    "chapters" (`chunks`) so each one fits comfortably.

3. **Embedding - write a numeric summary**
   For every chapter, write a list of numbers (a vector) that captures its    meaning. Two chapters with similar meanings get similar numbers.

4. **Storage - file the cards in a catalog**
   File every chapter together with its number-summary into an index. In    real projects we also **persist** that index to disk so we don't have    to re-embed everything every time we run our script.

The last search step (retrieval) is what your RAG chain will use later. This notebook completes steps 1 through 4 - including persistence - with real files.

---
## Setup

Shared configuration used by all three examples: a path finder, the same `RecursiveCharacterTextSplitter`, and the same local `OllamaEmbeddings`.

In [ ]:
from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS, Chroma
import shutil
import chromadb

# --- robust path to data_ingestion/ from anywhere ---
def find_data_dir():
    for root in (Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]):
        cand = root / "data_ingestion"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("Could not find data_ingestion folder")

DATA = find_data_dir()
print("Data dir:", DATA)

# --- one splitter + one embeddings model reused everywhere ---
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
emb      = OllamaEmbeddings(model="nomic-embed-text")  # 768 dims, local

# --- reset helpers so re-running cells does not duplicate or break data ---
def reset_index(folder):
    # FAISS: just remove the folder (read at load-time, no open handles)
    shutil.rmtree(folder, ignore_errors=True)
    print(f"  (cleared old {folder}/ if present)")

def reset_chroma(folder, collection):
    # Chroma keeps a live sqlite handle; delete the collection via its client
    # instead of deleting the folder, otherwise a re-run hits a readonly DB.
    try:
        chromadb.PersistentClient(path=folder).delete_collection(collection)
        print(f"  (deleted collection '{collection}' in {folder}/)")
    except Exception:
        pass  # collection did not exist yet

print("setup complete")

---
## Example 1: A Single Song (`heylog_12_gauge_lyrics.txt`)

The smallest document. We run all four steps on a single lyric file and store the chunks in **FAISS**, then save it to disk and reload it.

In [ ]:

# ---- STEP 1: INGESTION ----
# read the file into a Document
doc_1 = TextLoader(str(DATA / "heylog_12_gauge_lyrics.txt")).load()
print("Step 1 - documents:", len(doc_1))
print("  metadata:", doc_1[0].metadata)
print("  content chars:", len(doc_1[0].page_content))

In [ ]:

# ---- STEP 2: TRANSFORMATION ----
# split into small chunks
chunks_1 = splitter.split_documents(doc_1)
print("Step 2 - chunks:", len(chunks_1))
print("  chunk sizes:", [len(c.page_content) for c in chunks_1][:6], "...")

In [ ]:

# ---- STEP 3: EMBEDDING ----
# one vector per chunk (Ollama nomic-embed-text, 768 dims)
vecs_1 = emb.embed_documents([c.page_content for c in chunks_1])
print("Step 3 - vectors:", len(vecs_1), "| dims each:", len(vecs_1[0]))

In [ ]:

# ---- STEP 4: STORAGE (in-memory) ----
# FAISS stores chunk text + vectors together
vs_1 = FAISS.from_documents(chunks_1, emb)
print("Step 4 - FAISS index size:", vs_1.index.ntotal)

In [ ]:

# ---- STEP 4b: PERSIST to disk + RELOAD ----
# save_local writes two files: index.faiss (vectors) + index.pkl (docs)
reset_index("vector_index_1_song")
vs_1.save_local("vector_index_1_song")
print("Saved to disk: vector_index_1_song/ ->",
      sorted(p.name for p in Path("vector_index_1_song").iterdir()))

# reload WITHOUT re-embedding - just read the files back
vs_1_re = FAISS.load_local(
    "vector_index_1_song",
    emb,
    allow_dangerous_deserialization=True,  # index.pkl is a pickle file
)
print("Reloaded from disk, index size:", vs_1_re.index.ntotal)

In [ ]:

# ---- RETRIEVAL from the reloaded store ----
results_1 = vs_1_re.similarity_search("dreaming of someone", k=3)
print("Top matches (from reloaded FAISS):")
for d in results_1:
    print("  -", d.page_content[:70].replace(chr(10), " "))

---
## Example 2: XML Book Catalog (`sample_catalog.xml`)

XML is loaded as plain text (the raw tags). We store the chunks in **Chroma**, persist to a local directory, reload without re-embedding, and add a metadata filter.

In [ ]:

# ---- STEP 1: INGESTION ----
# read the XML file as raw text
doc_2 = TextLoader(str(DATA / "sample_catalog.xml")).load()
print("Step 1 - documents:", len(doc_2))
print("  content head:", doc_2[0].page_content[:60].replace(chr(10), " "))

In [ ]:

# ---- STEP 2: TRANSFORMATION ----
chunks_2 = splitter.split_documents(doc_2)
print("Step 2 - chunks:", len(chunks_2))

In [ ]:

# ---- STEP 3: EMBEDDING ----
vecs_2 = emb.embed_documents([c.page_content for c in chunks_2])
print("Step 3 - vectors:", len(vecs_2), "| dims each:", len(vecs_2[0]))

In [ ]:

# ---- STEP 4: STORAGE + PERSIST ----
# persist_directory automatically saves Chroma to disk as a sqlite DB
reset_chroma("vector_index_2_xml", "xml_catalog")
vs_2 = Chroma.from_documents(
    chunks_2,
    emb,
    persist_directory="vector_index_2_xml",
    collection_name="xml_catalog",
)
print("Step 4 - Chroma count:", vs_2._collection.count())
print("Persisted to disk:", sorted(p.name for p in Path("vector_index_2_xml").iterdir())[:6])

In [ ]:

# ---- STEP 4b: RELOAD without re-embedding ----
# attach to the existing persisted DB by pointing at the folder
vs_2_re = Chroma(
    persist_directory="vector_index_2_xml",
    embedding_function=emb,
    collection_name="xml_catalog",
)
print("Reloaded from disk, count:", vs_2_re._collection.count())

In [ ]:

# ---- RETRIEVAL with metadata filter from reloaded store ----
# keep only the science-fiction chunk (a kind of "where" filter)
results_2 = vs_2_re.similarity_search("science fiction novel", k=2)
print("Semantic matches (reloaded Chroma):")
for d in results_2:
    print("  -", d.page_content[:55].replace(chr(10), " "))

---
## Example 3: The Full `eve` Album (`heylog_eve_album.txt`)

The largest document - all 7 songs plus interpretations (~13KB). This is where chunking and persistence matter most. We store in **both** FAISS **and** Chroma, persist both, and compare semantic results.

### The `eve` album (2024) - running through all 4 steps

Tracklist: gravel, into the wind, running shoes, empty roads she's beautiful, 12 gauge, blinded by the sun, paranoid.

In [ ]:

# ---- STEP 1: INGESTION ----
doc_3 = TextLoader(str(DATA / "heylog_eve_album.txt")).load()
print("Step 1 - album documents:", len(doc_3))
print("  content chars:", len(doc_3[0].page_content))

In [ ]:

# ---- STEP 2: TRANSFORMATION ----
# 13KB splits into many small chunks, each with its own meaning
chunks_3 = splitter.split_documents(doc_3)
print("Step 2 - chunks:", len(chunks_3))
for i, c in enumerate(chunks_3[:3]):
    print(f"  chunk {i}: {c.page_content[:45].replace(chr(10), ' ')}")
print("  ...")

In [ ]:

# ---- STEP 3: EMBEDDING ----
vecs_3 = emb.embed_documents([c.page_content for c in chunks_3])
print("Step 3 - vectors:", len(vecs_3), "| dims each:", len(vecs_3[0]))

In [ ]:

# ---- STEP 4a: STORE in FAISS & persist ----
vs_3_faiss = FAISS.from_documents(chunks_3, emb)
reset_index("vector_index_3_album_faiss")
vs_3_faiss.save_local("vector_index_3_album_faiss")
print("FAISS index size:", vs_3_faiss.index.ntotal, "-> saved to disk")

In [ ]:

# ---- STEP 4b: STORE in Chroma & persist ----
reset_chroma("vector_index_3_album_chroma", "eve_album")
vs_3_chroma = Chroma.from_documents(
    chunks_3,
    emb,
    persist_directory="vector_index_3_album_chroma",
    collection_name="eve_album",
)
print("Chroma collection size:", vs_3_chroma._collection.count(),
      "-> saved to disk")

In [ ]:

# ---- STEP 4c: RELOAD both from disk (no re-embedding) ----
faiss_re = FAISS.load_local(
    "vector_index_3_album_faiss",
    emb,
    allow_dangerous_deserialization=True,
)
chroma_re = Chroma(
    persist_directory="vector_index_3_album_chroma",
    embedding_function=emb,
    collection_name="eve_album",
)
print("Reloaded FAISS size:", faiss_re.index.ntotal)
print("Reloaded Chroma count:", chroma_re._collection.count())

In [ ]:

# ---- RETRIEVAL from reloaded stores (same question) ----
query = "feelings of shame and hiding like Adam and Eve in the forest"

faiss_hits = faiss_re.similarity_search(query, k=3)
chroma_hits = chroma_re.similarity_search(query, k=3)

print("=== FAISS top 3 ===")
for d in faiss_hits:
    print("  -", d.page_content[:65].replace(chr(10), " "))

print("\n=== CHROMA top 3 ===")
for d in chroma_hits:
    print("  -", d.page_content[:65].replace(chr(10), " "))

---
## Summary

We walked each file through the **same four steps**:

| Step | Tool | What it does |
|------|------|--------------|
| 1. Ingestion | `TextLoader` | raw file -> `Document` |
| 2. Transformation | `RecursiveCharacterTextSplitter` | `Document` -> many small chunks |
| 3. Embedding | `OllamaEmbeddings` (nomic-embed-text) | each chunk -> 768-dim vector |
| 4. Storage | `FAISS` / `Chroma` | chunk + vector stored AND **persisted to disk** |

**Why persist matters:** a vector store lives in RAM by default. If you close the notebook, it is gone and you must re-ingest + re-embed everything. Saving the index to disk lets you **reload without re-embedding**, which saves money (hundreds or thousands of embedding API calls) and time on large corpora.

**Two ways to persist:**
- **FAISS** - `save_local(folder)` writes `index.faiss` + `index.pkl`;
  reload with `load_local(folder, embeddings)`.
- **Chroma** - pass `persist_directory=...` when building; reload by
  pointing a new `Chroma(persist_directory=...)` at the same folder.

**Folder layout after this notebook:**
- `vector_index_1_song/` - FAISS, 31 chunks from one song
- `vector_index_2_xml/` - Chroma, 12 chunks from XML catalog
- `vector_index_3_album_faiss/` + `vector_index_3_album_chroma/` - 165 chunks

**Where this flow goes next:** the persisted vectors feed a **RAG** pipeline - a query is embedded, the closest chunks are retrieved, and injected into an LLM prompt to ground its answer in real data. That is the final step of Section 5.